Script to scan all song folders under a segmented dataset and report missing or empty "audio" subfolders.


In [1]:
import os
import argparse

In [2]:
# Jupyter Notebook Cell 1: Imports and base path setup
import os
import shutil

# Base directory for segmented dataset
base_dir = "/mnt/Aimir_HD/suno/segmented"

# Cell 2: Define scan & cleanup function (with safety check)
def scan_and_cleanup(base_dir, dry_run=True):
    """
    Scan each song folder under base_dir. 
    Delete folders that have no files in 'audio/' AND no 'midi/' folder.

    Safety assertion ensures no folder containing a 'midi/' subfolder is ever deleted.
    """
    to_delete = []
    for folder in os.listdir(base_dir):
        song_path = os.path.join(base_dir, folder)
        if not os.path.isdir(song_path):
            continue

        audio_path = os.path.join(song_path, "audio")
        midi_path  = os.path.join(song_path, "midi")

        # Determine if audio/ contains any regular files
        audio_empty = True
        if os.path.isdir(audio_path):
            for f in os.listdir(audio_path):
                if os.path.isfile(os.path.join(audio_path, f)):
                    audio_empty = False
                    break

        # Only mark for deletion if audio_empty AND no midi folder
        if audio_empty and not os.path.isdir(midi_path):
            to_delete.append(song_path)

    print(f"Found {len(to_delete)} folders to delete.")
    if dry_run:
        for p in to_delete[:10]:
            print("DRY-RUN -> would delete:", p)
    else:
        # Safety check: ensure none of these has a midi/ subfolder
        for p in to_delete:
            if os.path.isdir(os.path.join(p, "midi")):
                raise RuntimeError(f"Safety check failed! MIDI folder found in {p}")
        # Now delete
        for p in to_delete:
            shutil.rmtree(p)
        print("Deletion complete.")

    return to_delete



In [3]:
# Cell 3: Dry-run to verify
candidates = scan_and_cleanup(base_dir, dry_run=True)


Found 0 folders to delete.


In [ ]:
# deleted = scan_and_cleanup(base_dir, dry_run=False)
# print(len(deleted), "folders actually deleted.")

Found 6 folders to delete.
Deletion complete.
6 folders actually deleted.


In [1]:
# Cell 1: Directory setup
import os

audio_dir = "/mnt/Aimir_HD/suno/audio"


In [2]:
# Cell 2: Header‐check loop
invalid = []
total = 0

for fname in os.listdir(audio_dir):
    if not fname.lower().endswith(".mp3"):
        continue
    total += 1
    path = os.path.join(audio_dir, fname)
    try:
        # Read first 3 bytes
        with open(path, "rb") as f:
            header = f.read(3)
        # Valid MP3 starts with b"ID3" or frame-sync 0xFFEx
        if not (
            header.startswith(b"ID3")
            or (len(header) >= 2 and header[0] == 0xFF and (header[1] & 0xE0) == 0xE0)
        ):
            invalid.append((fname, header))
    except Exception as e:
        invalid.append((fname, f"ERROR: {e!s}"))


In [3]:
# Cell 3: Report results
print(f"Found {total} .mp3 files under {audio_dir!r}.")
print(f"{len(invalid)} files failed the header check:\n")
for fname, hdr in invalid[:10]:
    print(f" - {fname}: header = {hdr!r}")
if len(invalid) > 10:
    print(f" ... and {len(invalid)-10} more.")


Found 187203 .mp3 files under '/mnt/Aimir_HD/suno/audio'.
0 files failed the header check:



In [ ]:
# Cell 3: Transcode invalid MP3s in-place via ffmpeg
import os
import subprocess

# audio_dir and invalid list come from Cell 1/2
# For each invalid file, decode → WAV → re-encode to MP3
for fname, header in invalid:
    src = os.path.join(audio_dir, fname)
    tmp = src + ".tmp.wav"
    print(f"Transcoding {fname}: header={header!r}")
    # 1) Decode container (RIFF/MP4) → WAV
    subprocess.run([
        "ffmpeg", "-y", "-i", src,
        "-ac", "2", "-ar", "44100",
        tmp
    ], check=True)
    # 2) Encode WAV → MP3
    subprocess.run([
        "ffmpeg", "-y", "-i", tmp,
        "-codec:a", "libmp3lame", "-qscale:a", "2",
        src
    ], check=True)
    # 3) Remove temporary WAV
    os.remove(tmp)

print("Transcoding of invalid files complete.")
